In [ ]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# အတုအယောက် စီးပွားရေးဒေတာများ ဖန်တီးခြင်း
np.random.seed(42)
n_samples = 1000

# စီးပွားရေးအညွှန်းကိန်းများ ဖန်တီးခြင်း
data = {
    'GDP_Growth': np.random.normal(3.5, 1.2, n_samples),  # GDP ကြီးထွားနှုန်း (%)
    'Inflation_Rate': np.random.normal(2.5, 1.0, n_samples),  # ငွေကြေးဖောင်းပွမှုနှုန်း (%)
    'Unemployment_Rate': np.random.normal(5.0, 1.5, n_samples),  # အလုပ်လက်မဲ့နှုန်း (%)
    'Interest_Rate': np.random.normal(2.0, 0.8, n_samples),  # အတိုးနှုန်း (%)
    'Stock_Market_Return': np.random.normal(8.0, 4.0, n_samples)  # စတော့ဈေးကွက်အကျိုးအမြတ် (%)
}

# DataFrame ဖန်တီးခြင်း
df = pd.DataFrame(data)

# စျေးကွက်အခြေအနေ ခွဲခြားခြင်း (0: ကျဆင်းမှု, 1: တည်ငြိမ်မှု, 2: ကြီးထွားမှု)
conditions = [
    (df['GDP_Growth'] < 2.0) | (df['Unemployment_Rate'] > 7.0),
    (df['GDP_Growth'] >= 2.0) & (df['GDP_Growth'] <= 5.0) & (df['Unemployment_Rate'] <= 7.0),
    (df['GDP_Growth'] > 5.0) & (df['Unemployment_Rate'] < 5.0)
]

choices = [0, 1, 2]  # 0: Recession, 1: Stable, 2: Growth
df['Market_Condition'] = np.select(conditions, choices, default=1)

# ဒေတာများကို ကြည့်ရှုခြင်း
print("ဒေတာအချက်အလက်များ၏ ပထမဆုံး ၅ ခု:")
print(df.head())

print("\nစျေးကွက်အခြေအနေ အမျိုးအစားခွဲခြားမှု:")
print(df['Market_Condition'].value_counts())

# Features နှင့် Target variable ခွဲခြားခြင်း
X = df[['GDP_Growth', 'Inflation_Rate', 'Unemployment_Rate', 'Interest_Rate', 'Stock_Market_Return']]
y = df['Market_Condition']

# Training နှင့် Testing sets ခွဲခြားခြင်း
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Data scaling (GaussianNB သည် scaling လုပ်စရာမလိုပါသော်လည်း ပိုကောင်းမွန်စေရန်)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# GaussianNB Model ဖန်တီးခြင်းနှင့် Training
gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train)

# Predictions လုပ်ခြင်း
y_pred = gnb.predict(X_test_scaled)

# Model Performance စစ်ဆေးခြင်း
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, 
                           target_names=['Recession', 'Stable', 'Growth']))

# Feature importance analysis
feature_importance = np.exp(gnb.theta_).mean(axis=0)
features = X.columns

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importance, y=features)
plt.title('Feature Importance for Economic Forecasting')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

# နမူနာ ခန့်မှန်းချက်တစ်ခု
sample_data = np.array([[4.2, 2.1, 4.5, 1.8, 9.5]])  # GDP: 4.2%, Inflation: 2.1%, etc.
sample_scaled = scaler.transform(sample_data)
prediction = gnb.predict(sample_scaled)
prediction_proba = gnb.predict_proba(sample_scaled)

print(f"\nနမူနာ ခန့်မှန်းချက်:")
print(f"စီးပွားရေးအညွှန်းကိန်းများ: {sample_data[0]}")
print(f"ခန့်မှန်းထားသော စျေးကွက်အခြေအနေ: {['Recession', 'Stable', 'Growth'][prediction[0]]}")
print(f"ဖြစ်နိုင်ခြေအလားအလာများ:")
print(f"  Recession: {prediction_proba[0][0]:.4f}")
print(f"  Stable: {prediction_proba[0][1]:.4f}")
print(f"  Growth: {prediction_proba[0][2]:.4f}")

## လက်တွေ့အသုံးချနိုင်သော စီးပွားရေးခန့်မှန်းချက် system

In [ ]:
# လက်တွေ့အသုံးချနိုင်သော စီးပွားရေးခန့်မှန်းချက် system
class EconomicForecaster:
    def __init__(self):
        self.model = GaussianNB()
        self.scaler = StandardScaler()
        self.is_trained = False
        
    def train(self, economic_data, market_conditions):
        """
        economic_data: DataFrame containing economic indicators
        market_conditions: Array of market conditions (0, 1, 2)
        """
        # Data scaling
        X_scaled = self.scaler.fit_transform(economic_data)
        
        # Model training
        self.model.fit(X_scaled, market_conditions)
        self.is_trained = True
        
    def predict_market_condition(self, economic_indicators):
        """
        လက်ရှိစီးပွားရေးအညွှန်းကိန်းများအပေါ်အခြေခံ၍ ခန့်မှန်းချက်ထုတ်ခြင်း
        """
        if not self.is_trained:
            raise ValueError("Model ကို အရင်ဆုံး training လုပ်ပေးရန် လိုအပ်ပါသည်")
        
        # Data scaling
        indicators_scaled = self.scaler.transform([economic_indicators])
        
        # Prediction
        prediction = self.model.predict(indicators_scaled)[0]
        probabilities = self.model.predict_proba(indicators_scaled)[0]
        
        return {
            'prediction': prediction,
            'label': ['Recession', 'Stable', 'Growth'][prediction],
            'probabilities': {
                'Recession': probabilities[0],
                'Stable': probabilities[1],
                'Growth': probabilities[2]
            }
        }

# Example usage
forecaster = EconomicForecaster()

# Training data (လက်တွေ့တွင် historical data များဖြင့် အစားထိုးရမည်)
train_data = X_train.values
train_labels = y_train.values

# Model training
forecaster.train(train_data, train_labels)

# လက်ရှိစီးပွားရေးအခြေအနေဖြင့် ခန့်မှန်းချက်ထုတ်ခြင်း
current_economic_indicators = [3.8, 2.3, 5.2, 2.1, 7.8]  # GDP, Inflation, Unemployment, etc.
prediction = forecaster.predict_market_condition(current_economic_indicators)

print("စီးပွားရေးခန့်မှန်းချက် ရလဒ်များ:")
print(f"ခန့်မှန်းချက်: {prediction['label']}")
print("ဖြစ်နိုင်ခြေအလားအလာများ:")
for condition, prob in prediction['probabilities'].items():
    print(f"  {condition}: {prob:.4f}")

In [ ]:
# ခန့်မှန်းချက်ရလဒ်များကို visualization လုပ်ခြင်း
def plot_economic_forecast(results, actual_data):
    conditions = ['Recession', 'Stable', 'Growth']
    probabilities = [results['probabilities'][cond] for cond in conditions]
    
    plt.figure(figsize=(12, 5))
    
    # Probability plot
    plt.subplot(1, 2, 1)
    bars = plt.bar(conditions, probabilities, color=['red', 'yellow', 'green'])
    plt.title('Market Condition Probabilities')
    plt.ylabel('Probability')
    plt.ylim(0, 1)
    
    # Add probability values on bars
    for bar, prob in zip(bars, probabilities):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{prob:.3f}', ha='center', va='bottom')
    
    # Economic indicators radar plot
    plt.subplot(1, 2, 2)
    indicators = ['GDP', 'Inflation', 'Unemployment', 'Interest', 'Stock Return']
    values = actual_data
    
    # Normalize values for radar plot
    values_normalized = (values - np.min(values)) / (np.max(values) - np.min(values))
    
    angles = np.linspace(0, 2*np.pi, len(indicators), endpoint=False).tolist()
    values_normalized += values_normalized[:1]
    angles += angles[:1]
    
    ax = plt.subplot(1, 2, 2, polar=True)
    ax.plot(angles, values_normalized, 'o-', linewidth=2)
    ax.fill(angles, values_normalized, alpha=0.25)
    ax.set_thetagrids(np.degrees(angles[:-1]), indicators)
    ax.set_title('Economic Indicators Radar')
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()

# Visualization ကို run ခြင်း
plot_economic_forecast(prediction, current_economic_indicators)